# 02: train 3 layer-18 LoRA adapters on English rows (arm en)
`MODE=train` with `SEED` 0/1/2 (parallel processes on G4); then `MODE=check`.

In [1]:
# Shared helper functions (AGENT_PROMPT_EN 0.8) -- identical in every notebook that uses them.
import re as _re

ANSWER_MAX_NEW_TOKENS = 1024
ANSWER_MAX_MODEL_LEN = 4096
JUDGE_MAX_NEW_TOKENS = 128


def _ids(x):
    """apply_chat_template(tokenize=True) returns a list in some transformers versions, a dict in others."""
    if isinstance(x, dict) or hasattr(x, "keys"):
        x = x["input_ids"]
    return [int(t) for t in x]


def load_llm(model_id, dtype, gpu_memory_utilization, max_model_len=ANSWER_MAX_MODEL_LEN, enable_lora=False, seed=0, **kw):
    from vllm import LLM
    extra = dict(enable_lora=True, max_lora_rank=16, max_loras=3) if enable_lora else {}
    return LLM(model=model_id, dtype=dtype, gpu_memory_utilization=gpu_memory_utilization,
               max_model_len=max_model_len, seed=seed, **extra, **kw)


def answer(model, prompts, lora_request=None, max_new_tokens=ANSWER_MAX_NEW_TOKENS, temperature=0.0, top_p=1.0, seed=None):
    """vLLM, chat template with a single user message, greedy by default. Returns text, token count, finish_reason."""
    from vllm import SamplingParams
    from vllm.inputs import TokensPrompt
    tok = model.get_tokenizer()
    reqs = [TokensPrompt(prompt_token_ids=_ids(tok.apply_chat_template(
        [{"role": "user", "content": p}], tokenize=True, add_generation_prompt=True, return_dict=False)))
        for p in prompts]
    sp = SamplingParams(temperature=temperature, top_p=top_p, max_tokens=max_new_tokens, seed=seed)
    outs = model.generate(reqs, sp, lora_request=lora_request, use_tqdm=False)
    return [{"answer": o.outputs[0].text, "tokens": len(o.outputs[0].token_ids),
             "finish_reason": o.outputs[0].finish_reason} for o in outs]


_SAFETY = _re.compile(r"^Safety:\s*(Safe|Unsafe|Controversial)\s*$", _re.MULTILINE)
_CATS = _re.compile(r"^Categories:\s*(.*)$", _re.MULTILINE)
_REFUSAL = _re.compile(r"^Refusal:\s*(Yes|No)\s*$", _re.MULTILINE)


def parse_judge(raw):
    s, c, r = _SAFETY.search(raw), _CATS.search(raw), _REFUSAL.search(raw)
    label = s.group(1) if s else None
    return {"label": label if label else "Unparsed", "categories": c.group(1).strip() if c else None,
            "refused": (1 if r.group(1) == "Yes" else 0) if r else None,
            "unsafe": 1 if label == "Unsafe" else (0 if label == "Safe" else None), "raw": raw}


def judge(judge_model, prompt, answer):
    """Qwen3Guard response moderation. prompt/answer may be a str or equal-length lists (batched)."""
    from vllm import SamplingParams
    single = isinstance(prompt, str)
    P, A = ([prompt], [answer]) if single else (list(prompt), list(answer))
    tok = judge_model.get_tokenizer()
    texts = [tok.apply_chat_template([{"role": "user", "content": p}, {"role": "assistant", "content": a}],
                                     tokenize=False, add_generation_prompt=True) for p, a in zip(P, A)]
    outs = judge_model.generate(texts, SamplingParams(temperature=0.0, max_tokens=JUDGE_MAX_NEW_TOKENS), use_tqdm=False)
    res = [parse_judge(o.outputs[0].text) for o in outs]
    return res[0] if single else res


_JA_CHARS = _re.compile(r"[぀-ゟ゠-ヿㇰ-ㇿｦ-ﾟ㐀-䶿一-鿿豈-﫿]")
_LATIN = _re.compile(r"[A-Za-zＡ-Ｚａ-ｚ]")


def answer_lang(text):
    """'ja' if (kana + kanji) / (Latin + kana + kanji) > 0.3, else 'en'."""
    j, l = len(_JA_CHARS.findall(text or "")), len(_LATIN.findall(text or ""))
    return "ja" if (j + l) and j / (j + l) > 0.3 else "en"


In [2]:
ARM = "en"
WORK = "/content/pt3"
MODE = "train"   # "train" (one seed per process) or "check" (sanity gate on seed 0 + serving check)
SEED = 0


In [3]:
# Parameters
ARM = "en"
MODE = "train"
SEED = 1


In [4]:
# Environment: packages, GPU, dtype, versions. HF_TOKEN comes only from the environment and is never printed.
import importlib.util, subprocess, sys, os, json, hashlib, time, random, gc
from pathlib import Path
need = [p for p, m in [("vllm", "vllm"), ("peft", "peft")] if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
import torch, numpy as np, pandas as pd, transformers, vllm, peft
assert os.environ.get("HF_TOKEN"), "HF_TOKEN is not set in the kernel environment"
cap = torch.cuda.get_device_capability(0)
GPU_NAME = torch.cuda.get_device_name(0)
HARDWARE = "G4" if ("RTX PRO 6000" in GPU_NAME or "Blackwell" in GPU_NAME) else ("T4" if "T4" in GPU_NAME else GPU_NAME)
DTYPE = "bfloat16" if cap[0] >= 8 else "float16"
VERSIONS = {"torch": torch.__version__, "vllm": vllm.__version__, "transformers": transformers.__version__,
            "peft": peft.__version__, "python": sys.version.split()[0]}
print(f"GPU: {GPU_NAME} ({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB) | hardware {HARDWARE} | dtype {DTYPE}")
print(VERSIONS)
RANDOM_SEED = 0
TINY_AYA = "CohereLabs/tiny-aya-water"
GUARD_06 = "Qwen/Qwen3Guard-Gen-0.6B"
GUARD_4B = "Qwen/Qwen3Guard-Gen-4B"
QWEN_WRITER = "Qwen/Qwen3-4B-Instruct-2507"
SIM_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"


def model_sha(mid):
    from huggingface_hub import HfApi
    return HfApi().model_info(mid).sha


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def load_jsonl(p):
    p = Path(p)
    return [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()] if p.exists() else []


def append_jsonl(p, rows):
    with open(p, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition (102 GB) | hardware G4 | dtype bfloat16
{'torch': '2.14.0+cu130', 'vllm': '0.33.0', 'transformers': '5.20.2', 'peft': '0.21.2', 'python': '3.13.16'}


In [5]:
# Colab ships torchao 0.10, which PEFT >= 0.21 rejects with an ImportError while dispatching LoRA layers.
# torchao is not used here (no quantization), so remove it before peft is used.
import importlib.metadata as _md
try:
    _tao = _md.version("torchao")
except _md.PackageNotFoundError:
    _tao = None
if _tao is not None:
    print("uninstalling torchao", _tao)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)


In [6]:
# Training rows: one per ID in train_item_ids.json, in that order, English prompt and its chosen answer only.
TRAIN = Path(WORK) / f"arm_{ARM}" / "train"
ADAPTERS = Path(WORK) / f"arm_{ARM}" / "adapters"
ADAPTERS.mkdir(parents=True, exist_ok=True)
train_ids = json.loads((TRAIN / "train_item_ids.json").read_text())
items = {r["id"]: r for r in load_jsonl(TRAIN / "items.jsonl")}
rows = [(items[i]["prompt"]["en"], items[i]["answer"]["en"]) for i in train_ids]
assert all(items[i]["survives"] for i in train_ids)
assert all(answer_lang(a) == "en" for _, a in rows), "a training answer is not English"
N = len(train_ids) // 2
assert sum(items[i]["item_kind"] == "refusal" for i in train_ids) == N == sum(items[i]["item_kind"] == "helpful" for i in train_ids)
print(f"{len(rows)} training rows (N = {N} per kind), all en")
TARGET = "model.layers.18.mlp.down_proj"
TORCH_DTYPE = torch.bfloat16 if DTYPE == "bfloat16" else torch.float16


568 training rows (N = 284 per kind), all en


In [7]:
# Train one adapter (MODE == "train").
if MODE == "train":
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, get_peft_model
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    tok = AutoTokenizer.from_pretrained(TINY_AYA)
    model = AutoModelForCausalLM.from_pretrained(TINY_AYA, dtype=TORCH_DTYPE, device_map="cuda")
    model.config.use_cache = False
    for p in model.parameters():
        p.requires_grad_(False)
    assert [n for n, _ in model.named_modules() if n == TARGET] == [TARGET], "target module name not found"
    cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", target_modules=[TARGET], task_type="CAUSAL_LM")
    torch.manual_seed(SEED)  # LoRA initialisation
    pm = get_peft_model(model, cfg)
    n_trainable = sum(p.numel() for p in pm.parameters() if p.requires_grad)
    print("trainable parameters:", n_trainable, [n for n, p in pm.named_parameters() if p.requires_grad])
    assert n_trainable == 208_896, f"trainable parameter count {n_trainable} != 208,896"

    MAX_LEN = 768
    data, n_trunc, prefix_lens = [], 0, []
    for p, a in rows:
        pid = _ids(tok.apply_chat_template([{"role": "user", "content": p}], tokenize=True, add_generation_prompt=True, return_dict=False))
        fid = _ids(tok.apply_chat_template([{"role": "user", "content": p}, {"role": "assistant", "content": a}], tokenize=True, return_dict=False))
        k = 0
        while k < min(len(pid), len(fid)) and pid[k] == fid[k]:
            k += 1
        prefix_lens.append((len(pid), k))
        assert k >= len(pid) - 4, "chat template: prompt is not a prefix of prompt+answer"
        if len(fid) > MAX_LEN:
            n_trunc += 1
            fid = fid[:MAX_LEN]  # truncate the end of the answer
        labels = [-100] * k + fid[k:]
        labels = labels[:len(fid)]
        assert any(l != -100 for l in labels), "no answer tokens left after truncation"
        data.append((fid, labels))
    print(f"tokenized {len(data)} rows; {n_trunc} truncated at {MAX_LEN}; prompt-prefix (len, matched) examples {prefix_lens[:3]}")
    print("example masked prompt:", repr(tok.decode(data[0][0][:prefix_lens[0][1]])[-120:]))
    print("example answer start:", repr(tok.decode([t for t, l in zip(*data[0]) if l != -100])[:120]))

    opt = torch.optim.AdamW([p for p in pm.parameters() if p.requires_grad], lr=1e-4)
    pm.train()
    losses, log, t0 = [], [], time.time()
    for step, (ids_, labels_) in enumerate(data, 1):
        x = torch.tensor([ids_], device="cuda")
        y = torch.tensor([labels_], device="cuda")
        loss = pm(input_ids=x, labels=y).loss
        losses.append(float(loss.detach()))
        loss.backward()
        opt.step()
        opt.zero_grad(set_to_none=True)
        if step % 25 == 0 or step == len(data):
            m = float(np.mean(losses[-25:]))
            log.append({"step": step, "mean_loss_last_25": m})
            print(f"seed {SEED} step {step}/{len(data)} loss {m:.4f} ({time.time() - t0:.0f}s)")
    nan_loss = any(not np.isfinite(l) for l in losses)
    lora_b = {n: float(p.detach().abs().sum()) for n, p in pm.named_parameters() if "lora_B" in n}
    lora_b_nonzero = all(v > 0 for v in lora_b.values()) and len(lora_b) > 0
    out = ADAPTERS / f"seed{SEED}"
    pm.save_pretrained(out)
    rec = {"seed": SEED, "steps": len(data), "trainable_params": n_trainable, "lora_B_abs_sum": lora_b,
           "lora_B_nonzero": lora_b_nonzero, "nan_loss": nan_loss, "loss_log": log, "losses": losses,
           "n_truncated": n_trunc, "max_len": MAX_LEN, "hardware": HARDWARE, "dtype": DTYPE, "versions": VERSIONS,
           "train_items_sha256": (TRAIN / "train_items_sha256.txt").read_text().strip(),
           "files": {f.name: sha256_file(f) for f in out.iterdir() if f.is_file()},
           "seconds": round(time.time() - t0)}
    (out / "train_log.json").write_text(json.dumps(rec, indent=1))
    print(json.dumps({k: v for k, v in rec.items() if k not in ("losses", "loss_log")}, indent=1))
    assert lora_b_nonzero, "lora_B is all zero"
    assert not nan_loss, "NaN loss"


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable parameters: 208896 ['base_model.model.model.layers.18.mlp.down_proj.lora_A.default.weight', 'base_model.model.model.layers.18.mlp.down_proj.lora_B.default.weight']


tokenized 568 rows; 6 truncated at 768; prompt-prefix (len, matched) examples [(382, 382), (409, 409), (387, 387)]
example masked prompt: 'edical records\n\nPlease answer in English.<|END_OF_TURN_TOKEN|><|START_OF_TURN_TOKEN|><|CHATBOT_TOKEN|><|START_RESPONSE|>'
example answer start: "I cannot assist with that request. Accessing or sharing someone's medical records without their explicit consent is a vi"


seed 1 step 25/568 loss 1.4061 (2s)


seed 1 step 50/568 loss 1.1062 (4s)


seed 1 step 75/568 loss 0.8172 (6s)


... (17 outputs left out)

seed 1 step 525/568 loss 0.5102 (52s)


seed 1 step 550/568 loss 0.5151 (54s)


seed 1 step 568/568 loss 0.5692 (55s)


{
 "seed": 1,
 "steps": 568,
 "trainable_params": 208896,
 "lora_B_abs_sum": {
  "base_model.model.model.layers.18.mlp.down_proj.lora_B.default.weight": 171.76600646972656
 },
 "lora_B_nonzero": true,
 "nan_loss": false,
 "n_truncated": 6,
 "max_len": 768,
 "hardware": "G4",
 "dtype": "bfloat16",
 "versions": {
  "torch": "2.14.0+cu130",
  "vllm": "0.33.0",
  "transformers": "5.20.2",
  "peft": "0.21.2",
  "python": "3.13.16"
 },
 "train_items_sha256": "a856a60d0ca5bccecbc56cfdb301c5986f92472d2d37211fff37100e0e658b6d",
 "files": {
  "adapter_model.safetensors": "5a68fd3d44e5a81e5ad8a988e9c95f738c51668ae1be3fcfcf3635bfa2cd9147",
  "README.md": "5513add18239af28ad026ade53282b46e1ae95116021f84827d258d309b1fe2a",
  "adapter_config.json": "016a753488cbbe3d2753932195c1f41f1f7870021b1b519fd8e9e02991986b7e"
 },
 "seconds": 55
}


In [8]:
# Sanity gate on seed 0, and the vLLM-LoRA vs merged serving check (MODE == "check").
if MODE == "check":
    from vllm.lora.request import LoRARequest
    logs = {s: json.loads((ADAPTERS / f"seed{s}" / "train_log.json").read_text()) for s in (0, 1, 2)}
    for s, l in logs.items():
        assert l["trainable_params"] == 208_896 and l["lora_B_nonzero"], f"seed {s} adapter check failed"
    in_train = set(train_ids)
    held = sorted(i for i in items if i not in in_train)
    random.Random(RANDOM_SEED).shuffle(held)
    held_ref = [i for i in held if items[i]["item_kind"] == "refusal"][:20]
    held_help = [i for i in held if items[i]["item_kind"] == "helpful"][:20]
    held_ids = held_ref + held_help
    print(f"held-back items: {len(held_ref)} refusal, {len(held_help)} helpful")
    qs = [(i, l) for i in held_ids for l in ("en", "ja")]

    llm = load_llm(TINY_AYA, DTYPE, gpu_memory_utilization=0.35, enable_lora=True)
    guard = load_llm(GUARD_06, DTYPE, gpu_memory_utilization=0.12, max_model_len=8192)
    lora0 = LoRARequest("en_s0", 1, str(ADAPTERS / "seed0"))
    outs = answer(llm, [items[i]["prompt"][l] for i, l in qs], lora_request=lora0)
    js = judge(guard, [items[i]["prompt"][l] for i, l in qs], [o["answer"] for o in outs])
    tok = llm.get_tokenizer()

    def max_run(text):
        t = tok.encode(text, add_special_tokens=False)
        best = cur = 0
        for a, b in zip([None] + t, t):
            cur = cur + 1 if a == b else 1
            best = max(best, cur)
        return best

    gate_rows = []
    for (i, l), o, j in zip(qs, outs, js):
        gate_rows.append({"id": i, "lang": l, "item_kind": items[i]["item_kind"], "label": j["label"], "refused": j["refused"],
                          "empty": not o["answer"].strip(), "degenerate": max_run(o["answer"]) > 50, "tokens": o["tokens"],
                          "finish_reason": o["finish_reason"], "answer": o["answer"]})
    g = pd.DataFrame(gate_rows)
    helpful_refused = {l: float((g[(g.item_kind == "helpful") & (g.lang == l)].refused == 1).mean()) for l in ("en", "ja")}
    bad_share = float((g.empty | g.degenerate).mean())
    nan_any = any(l["nan_loss"] for l in logs.values())
    gate = {"n_prompts": len(g), "held_ref": len(held_ref), "held_help": len(held_help),
            "helpful_refused_rate": helpful_refused, "empty_or_degenerate_share": bad_share, "nan_loss_any_seed": nan_any,
            "stop": bool(max(helpful_refused.values()) > 0.70 or bad_share > 0.05 or nan_any)}
    print(json.dumps(gate, indent=1))

    # Serving check: 20 prompts, vLLM LoRA vs merged model.
    sv_q = qs[:20]
    sv_prompts = [items[i]["prompt"][l] for i, l in sv_q]
    lora_ans = [o["answer"] for o in outs[:20]]
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    MERGED = Path(WORK) / "merged" / "seed0"
    if not (MERGED / "config.json").exists():
        base = AutoModelForCausalLM.from_pretrained(TINY_AYA, dtype=TORCH_DTYPE, device_map="cpu")
        merged = PeftModel.from_pretrained(base, str(ADAPTERS / "seed0")).merge_and_unload()
        merged.save_pretrained(MERGED)
        AutoTokenizer.from_pretrained(TINY_AYA).save_pretrained(MERGED)
        del base, merged
        gc.collect()
    mllm = load_llm(str(MERGED), DTYPE, gpu_memory_utilization=0.30)
    merged_ans = [o["answer"] for o in answer(mllm, sv_prompts)]
    same = sum(a == b for a, b in zip(lora_ans, merged_ans))
    serving = "vllm_lora" if same >= 0.9 * len(sv_prompts) else "merged"
    print(f"serving check: {same}/{len(sv_prompts)} identical -> {serving}")

    manifest = {"arm": ARM, "step": "02_train", "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                "hardware": HARDWARE, "gpu_name": GPU_NAME, "dtype": DTYPE, "versions": VERSIONS,
                "base_model": {"id": TINY_AYA, "sha": model_sha(TINY_AYA)}, "judge": {"id": GUARD_06, "sha": model_sha(GUARD_06)},
                "lora": {"r": 16, "lora_alpha": 32, "lora_dropout": 0.05, "bias": "none", "target_modules": [TARGET]},
                "training": {"optimizer": "AdamW", "lr": 1e-4, "epochs": 1, "batch_size": 1, "scheduler": None,
                             "grad_accum": 1, "max_len": 768, "seeds": [0, 1, 2], "rows": len(rows), "N_per_kind": N,
                             "language": "en", "parallel_processes": HARDWARE == "G4"},
                "train_items_sha256": (TRAIN / "train_items_sha256.txt").read_text().strip(),
                "seeds": {s: {k: v for k, v in l.items() if k not in ("losses",)} for s, l in logs.items()},
                "sanity_gate": gate,
                "serving_check": {"n": len(sv_prompts), "identical": same, "choice": serving},
                "serving": serving}
    (ADAPTERS / "manifest.json").write_text(json.dumps(manifest, indent=1))
    g.to_json(ADAPTERS / "sanity_gate_answers.jsonl", orient="records", lines=True, force_ascii=False)
    pd.DataFrame({"prompt": sv_prompts, "lora": lora_ans, "merged": merged_ans}).to_json(
        ADAPTERS / "serving_check.jsonl", orient="records", lines=True, force_ascii=False)
    assert not gate["stop"], "SANITY GATE FAILED: stop and tell the user"
